In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from IPython.display import display

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})

DB_PATH = Path("..") / "results" / "optuna_studies.db"
STORAGE  = f"sqlite:///{DB_PATH.resolve()}"
print(f"DB: {DB_PATH.resolve()}  |  exists: {DB_PATH.exists()}")

In [ ]:
# Decode study name → (dataset, data_mode, model_type)
_RE = re.compile(
    r"^(?P<dataset>.+)_(?P<data_mode>clean|ar|nar)_(?P<model_type>linear|mlp)"
    r"(?:_curr[01]_gate[01]|_ag|_saga|_cp)$"
)

rows = []
for name in optuna.get_all_study_names(storage=STORAGE):
    m = _RE.match(name)
    if not m:
        continue
    study = optuna.load_study(study_name=name, storage=STORAGE)
    try:
        best = study.best_trial
    except Exception:
        continue  # no completed trials yet

    # Retrieve per-seed test metrics stored by _objective()
    seed_results = best.user_attrs.get("seed_results", [])
    test_f1_values = [
        r["test_f1"] for r in seed_results
        if "test_f1" in r and r["test_f1"] is not None
    ]

    if not test_f1_values:
        # Fall back to the aggregated attribute written by _objective()
        mean_f1 = best.user_attrs.get("avg_test_metric")
        std_f1  = best.user_attrs.get("std_test_metric", 0.0)
    else:
        mean_f1 = float(np.mean(test_f1_values))
        std_f1  = float(np.std(test_f1_values))

    if mean_f1 is None:
        continue

    rows.append({
        **m.groupdict(),
        "study_name": name,
        "best_val":   best.value,
        "test_f1_mean": mean_f1,
        "test_f1_std":  std_f1,
        "n_seeds": len(test_f1_values),
    })

df = pd.DataFrame(rows)
print(f"{len(df)} studies with results.")
df.head()

In [ ]:
# For each (dataset, data_mode, model_type) keep the config with highest test_f1_mean
best = (
    df.sort_values("test_f1_mean", ascending=False)
      .groupby(["dataset", "data_mode", "model_type"], sort=False)
      .first()
      .reset_index()
)

summary = best[["dataset", "data_mode", "model_type",
                "test_f1_mean", "test_f1_std", "n_seeds"]].sort_values(
    ["dataset", "model_type", "data_mode"]
).reset_index(drop=True)

display(
    summary.style
    .format({"test_f1_mean": "{:.4f}", "test_f1_std": "{:.4f}"})
    .background_gradient(subset=["test_f1_mean"], cmap="YlGn")
    .set_caption("Best test F1 per (dataset, data_mode, model_type)")
)

In [ ]:
DATA_MODE_COLORS = {"clean": "#4c9bcd", "ar": "#e07b39", "nar": "#7cb87c"}
DATA_MODES = [m for m in ("clean", "ar", "nar") if m in summary["data_mode"].unique()]

for model_type in sorted(summary["model_type"].unique()):
    sub = summary[summary["model_type"] == model_type]
    datasets = sorted(sub["dataset"].unique())

    x = np.arange(len(datasets))
    width = 0.8 / len(DATA_MODES)

    fig, ax = plt.subplots(figsize=(max(10, len(datasets) * 0.9), 5))

    for i, mode in enumerate(DATA_MODES):
        mode_data = sub[sub["data_mode"] == mode].set_index("dataset")
        means = [mode_data.loc[d, "test_f1_mean"] if d in mode_data.index else np.nan
                 for d in datasets]
        stds  = [mode_data.loc[d, "test_f1_std"]  if d in mode_data.index else 0.0
                 for d in datasets]
        offset = (i - len(DATA_MODES) / 2 + 0.5) * width
        ax.bar(x + offset, means, width=width * 0.9,
               yerr=stds, capsize=3,
               color=DATA_MODE_COLORS[mode], label=mode,
               error_kw={"elinewidth": 1, "ecolor": "#444"})

    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Test F1 (mean ± std)")
    ax.set_title(f"Best test F1 by dataset and data mode — {model_type.upper()}")
    ax.set_ylim(0, 1.05)
    ax.legend(title="data_mode")
    ax.grid(axis="y", linewidth=0.4, alpha=0.5)
    plt.tight_layout()
    plt.show()